In [1]:
!pip install pyspark

In [6]:
import os
import shutil
import glob

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    initcap,
    when,
    sum,
    count,
    isnan,
    to_date,
    coalesce
)

In [7]:
spark = SparkSession.builder \
    .appName("SalesDataLake") \
    .getOrCreate()

print("Spark started successfully!")
print("Spark version:", spark.version)

Spark started successfully!
Spark version: 4.0.4


In [5]:
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

print("Data Lake folders created.")

In [8]:
bronze_path = "data_lake/bronze/sales.csv"

print("Bronze file exists:", os.path.exists(bronze_path))
print("Bronze file path:", bronze_path)

Bronze file exists: True
Bronze file path: data_lake/bronze/sales.csv


In [9]:
df = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Bronze data loaded successfully.")

Bronze data loaded successfully.


In [10]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [11]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [12]:
bronze_count = df.count()

print("Total Bronze records:", bronze_count)

Total Bronze records: 1000


In [13]:
print("Columns:")
for column in df.columns:
    print("-", column)

Columns:
- order_id
- customer_id
- customer_name
- product
- category
- quantity
- unit_price
- discount_percent
- payment_method
- city
- state
- order_date
- order_status


In [14]:
total_rows = df.count()
distinct_rows = df.distinct().count()

duplicate_count = total_rows - distinct_rows

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicate rows:", duplicate_count)

Total rows: 1000
Distinct rows: 990
Exact duplicate rows: 10


In [15]:
duplicates = df.groupBy(df.columns).count().filter(col("count") > 1)

print("Duplicate groups:")
duplicates.show(truncate=False)

Duplicate groups:
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Groce

In [16]:
duplicates = df.groupBy(df.columns).count().filter(col("count") > 1)

print("Duplicate groups:")
duplicates.show(truncate=False)

Duplicate groups:
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Groce

In [18]:
print("Null values in each column:")

df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

Null values in each column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [19]:
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    print("\n==========", c, "==========")
    df.select(c).distinct().show(30, truncate=False)


========== customer_name ==========
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |
|Pooja Jain        |
|Siddharth Kapoor  |
|Keerthi Tripathi  |
|Aditya Chopra     |
|Arjun Agarwal     |
|Vivaan Nair       |
|Deepak Pandey     |
|Pooja Verma       |
|Neha Sharma       |
|Divya Chopra      |
|Aarav Nair        |
|Kavya Malhotra    |
+------------------+
only showing top 30 rows

========== product ==========
+----------------------+
|product               |
+----------------------+
|Power Bank            |
|Wireless Mouse        |
|Dumbbell Set          |
|Mixer Grinder         |
|Jacket           

In [20]:
invalid_quantity = df.filter(
    col("quantity") <= 0
)

print("Invalid quantity records:", invalid_quantity.count())

invalid_quantity.show(truncate=False)

Invalid quantity records: 12
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100114  |C0252      |Keerthi Singh|Ceiling Fan           |Home Appliances|0       |2920      |15              |Net Banking     |Chennai   |Tamil Nadu    |2025-02-15|Delivered   |
|100143  |C0184      |Aman Joshi   |Football              |Sports         |0       |1030      |15              |Debit Card      |Coimbatore|Tamil Nadu    |2025-11-26|Delivered   |
|100198  |C0014      |Arjun Pillai |Hoodie                |Clothing    

In [21]:
invalid_price = df.filter(
    col("unit_price") <= 0
)

print("Invalid unit price records:", invalid_price.count())

invalid_price.show(truncate=False)

Invalid unit price records: 8
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery    |7       |0         |40              |Credit Card   |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100170  |C0158      |Deepak Pandey   |Competitive Exam Guide|Books      |6       |0         |0               |Net Banking   |Pune      |Maharashtra   |2025-08-26|Shipped     |
|100235  |C0349      |Tanvi Reddy     |Power Bank            |Electronics|7       |-5

In [22]:
invalid_discount = df.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
)

print("Invalid discount records:", invalid_discount.count())

invalid_discount.show(truncate=False)

Invalid discount records: 8
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|customer_name  |product         |category       |quantity|unit_price|discount_percent|payment_method  |city      |state     |order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|100034  |C0192      |Divya Sharma   |Yoga Mat        |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|Delivered   |
|100140  |C0075      |Swathi Tripathi|Cooking Oil 5L  |Grocery        |4       |790       |-5              |Credit Card     |Hyderabad |Telangana |2025-10-01|Shipped     |
|100217  |C0214      |Neha Tripathi  |Football        |Sports         |1       |650       |-5              |UPI 

In [29]:
df.select("order_date").show(20, truncate=False)

+----------+
|order_date|
+----------+
|2026-01-01|
|2025-08-03|
|2025-05-06|
|2026-07-31|
|2026-08-30|
|2026-05-09|
|2025-12-02|
|2025-09-25|
|2025-09-04|
|2026-08-24|
|2026-07-27|
|2026-09-18|
|2025-08-16|
|2025-12-20|
|2026-06-30|
|2025-04-08|
|2026-05-25|
|2026-08-17|
|2026-05-11|
|2025-12-06|
+----------+
only showing top 20 rows


In [31]:
from pyspark.sql.functions import col, trim, expr

date_check = df.withColumn(
    "parsed_date",
    expr("try_cast(trim(order_date) AS DATE)")
)

invalid_dates = date_check.filter(
    col("parsed_date").isNull() & col("order_date").isNotNull()
)

print("Invalid date records:", invalid_dates.count())

invalid_dates.select(
    "order_id",
    "order_date"
).show(truncate=False)

Invalid date records: 6
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+



In [32]:
df_silver = df.dropDuplicates()

print("Rows before duplicate removal:", df.count())
print("Rows after duplicate removal:", df_silver.count())

Rows before duplicate removal: 1000
Rows after duplicate removal: 990


In [33]:
df_silver = df.dropDuplicates()

print("Rows before duplicate removal:", df.count())
print("Rows after duplicate removal:", df_silver.count())

Rows before duplicate removal: 1000
Rows after duplicate removal: 990


In [34]:
for c in text_columns:
    df_silver = df_silver.withColumn(
        c,
        trim(col(c))
    )

print("Extra spaces removed from text columns.")

Extra spaces removed from text columns.


In [35]:
for c in text_columns:
    df_silver = df_silver.withColumn(
        c,
        trim(col(c))
    )

print("Extra spaces removed from text columns.")

Extra spaces removed from text columns.


In [36]:
df_silver = df_silver.withColumn(
    "customer_name",
    initcap(trim(col("customer_name")))
)

print("Customer names standardized.")

Customer names standardized.


In [37]:
df_silver = df_silver.withColumn(
    "quantity",
    when(
        col("quantity") > 0,
        col("quantity")
    ).otherwise(None)
)

print("Invalid quantities converted to NULL.")

Invalid quantities converted to NULL.


In [38]:
median_quantity = df_silver.approxQuantile(
    "quantity",
    [0.5],
    0.01
)[0]

print("Median quantity:", median_quantity)

df_silver = df_silver.fillna({
    "quantity": median_quantity
})

Median quantity: 5.0


In [39]:
df_silver = df_silver.withColumn(
    "unit_price",
    when(
        col("unit_price") > 0,
        col("unit_price")
    ).otherwise(None)
)

print("Invalid unit prices converted to NULL.")

Invalid unit prices converted to NULL.


In [40]:
median_price = df_silver.approxQuantile(
    "unit_price",
    [0.5],
    0.01
)[0]

print("Median unit price:", median_price)

df_silver = df_silver.fillna({
    "unit_price": median_price
})

Median unit price: 1810.0


In [42]:
df_silver = df_silver.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

print("Invalid discount percentages converted to NULL.")

Invalid discount percentages converted to NULL.


In [43]:
df_silver = df_silver.fillna({
    "discount_percent": 0
})

print("Missing discounts filled with 0.")

Missing discounts filled with 0.


In [44]:
df_silver = df_silver.withColumn(
    "order_date",
    to_date(
        trim(col("order_date")),
        "yyyy-MM-dd"
    )
)

print("Order dates converted to yyyy-MM-dd format.")

Order dates converted to yyyy-MM-dd format.


In [47]:
invalid_date_count = df_silver.filter(
    col("order_date").isNull()
).count()

print("Invalid/missing dates after conversion:", invalid_date_count)

Invalid/missing dates after conversion: 6


In [48]:
df_silver = df_silver.fillna({
    "customer_name": "UNKNOWN",
    "product": "UNKNOWN",
    "category": "UNKNOWN",
    "payment_method": "UNKNOWN",
    "city": "UNKNOWN",
    "state": "UNKNOWN",
    "order_status": "UNKNOWN"
})

print("Missing text values handled.")

Missing text values handled.


In [49]:
df_silver.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)



In [50]:
df_silver.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [51]:
silver_count = df_silver.count()

print("================================")
print("PREPROCESSING SUMMARY")
print("================================")
print("Bronze records:", bronze_count)
print("Silver records:", silver_count)
print("Records removed:", bronze_count - silver_count)

PREPROCESSING SUMMARY
Bronze records: 1000
Silver records: 1000
Records removed: 0


In [52]:
silver_spark_path = "data_lake/silver/preprocessed_sales"

df_silver.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_spark_path)

print("Silver data saved successfully.")

Silver data saved successfully.


In [53]:
print(os.listdir(silver_spark_path))

['part-00000-bd3cc166-6e67-4a8f-8437-4bb741bbd01e-c000.csv', '._SUCCESS.crc', '_SUCCESS', '.part-00000-bd3cc166-6e67-4a8f-8437-4bb741bbd01e-c000.csv.crc']


In [54]:
single_csv_path = "data_lake/silver/sales_silver_temp"

df_silver.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(single_csv_path)

print("Single CSV created.")

Single CSV created.


In [55]:
part_files = glob.glob(
    single_csv_path + "/part-*.csv"
)

print("Part files found:", part_files)

shutil.copy(
    part_files[0],
    "data_lake/silver/sales_silver.csv"
)

print("Created: data_lake/silver/sales_silver.csv")

Part files found: ['data_lake/silver/sales_silver_temp/part-00000-ebeb58ba-7a00-4835-97be-9ae2ba48d31c-c000.csv']
Created: data_lake/silver/sales_silver.csv


In [56]:
silver_df = spark.read.csv(
    "data_lake/silver/sales_silver.csv",
    header=True,
    inferSchema=True
)

print("Silver data read successfully.")

Silver data read successfully.


In [57]:
print("Silver record count:", silver_df.count())

print("\nSilver sample:")
silver_df.show(5, truncate=False)

print("\nSilver schema:")
silver_df.printSchema()

Silver record count: 1000

Silver sample:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-

In [58]:
print("========================================")
print("FINAL DATA LAKE VERIFICATION")
print("========================================")

print("Bronze record count :", bronze_count)
print("Silver record count :", silver_df.count())
print("Duplicate records removed :", duplicate_count)

print("\nSilver sample:")
silver_df.show(5, truncate=False)

print("\nSilver columns:")
print(silver_df.columns)

FINAL DATA LAKE VERIFICATION
Bronze record count : 1000
Silver record count : 1000
Duplicate records removed : 10

Silver sample:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clot